# 03 · A/B 实验框架分析：标题数字表达是否提升传播效果？

## 实验设计

| 项目 | 内容 |
|---|---|
| 业务假设 | 标题中加入数字表达（如"3个方法""5分钟看懂"）能提升视频播放量 |
| 实验组（T） | 标题含数字 `title_has_number=1`（2207 条） |
| 对照组（C） | 标题不含数字 `title_has_number=0`（1922 条） |
| 核心指标 | 播放量对数 `log_view`（近似正态、对极端值稳健） |
| 统计方法 | 均值差与 95% CI、Welch t 检验、Mann-Whitney U、Cohen's d、MDE |

**关键前提**：本数据是**观察性样本而非随机分流实验**，本 Notebook 会演示：
**朴素对比如何被混杂因素反转（辛普森悖论）→ 如何控制混杂 → 真实 A/B 实验的正确做法**。

In [1]:
import sys
from pathlib import Path
if (Path.cwd() / 'src').exists():
    sys.path.insert(0, str(Path.cwd()))
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from src import paths

df = pd.read_csv(paths.MODELING_CSV, encoding='utf-8-sig', low_memory=False)
t = df.loc[df['title_has_number']==1, 'log_view'].dropna()
c = df.loc[df['title_has_number']==0, 'log_view'].dropna()
n_t, n_c = len(t), len(c)
print(f'实验组(含数字) n={n_t}，对照组(不含数字) n={n_c}')
print(f'实验组均值={t.mean():.4f}，对照组均值={c.mean():.4f}')

实验组(含数字) n=2207，对照组(不含数字) n=1922
实验组均值=11.0783，对照组均值=11.1869


In [2]:
# 第一步：朴素组间比较（观察性数据直接比较，结果可能是"假象"）
mean_diff = t.mean() - c.mean()
se = np.sqrt(t.var(ddof=1)/n_t + c.var(ddof=1)/n_c)
df_w = (t.var(ddof=1)/n_t + c.var(ddof=1)/n_c)**2 / (
    (t.var(ddof=1)/n_t)**2/(n_t-1) + (c.var(ddof=1)/n_c)**2/(n_c-1))
ci = mean_diff + stats.t.ppf([0.025,0.975], df_w) * se
t_stat, p_t = stats.ttest_ind(t, c, equal_var=False)
u_stat, p_u = stats.mannwhitneyu(t, c, alternative='two-sided')
pooled = np.sqrt(((n_t-1)*t.var(ddof=1)+(n_c-1)*c.var(ddof=1))/(n_t+n_c-2))
d = mean_diff/pooled
print(f'朴素均值差 = {mean_diff:.4f}，95% CI [{ci[0]:.4f}, {ci[1]:.4f}]')
print(f'Welch t={t_stat:.3f}, p={p_t:.3f}； Mann-Whitney p={p_u:.3f}； Cohen d={d:.3f}')
print('\n注意：朴素结论是含数字标题反而"更差"。但这是真实的因果效应吗？——需要检查混杂。')

朴素均值差 = -0.1086，95% CI [-0.2627, 0.0455]
Welch t=-1.382, p=0.167； Mann-Whitney p=0.314； Cohen d=-0.043

注意：朴素结论是含数字标题反而"更差"。但这是真实的因果效应吗？——需要检查混杂。


In [3]:
# 第二步：混杂诊断。数字标题在不同领域的使用率差别巨大，而领域本身播放水平不同
usage = df.groupby('domain').agg(
    数字标题占比=('title_has_number', lambda s: round(s.mean()*100,1)),
    领域平均播放对数=('log_view', 'mean'),
    视频数=('bvid','count')).sort_values('数字标题占比', ascending=False)
display(usage.round(2))
print('解读：播放量最低的职场教育领域数字标题占比高达 79.4%，而高播放的科学科普仅 51.1%——')
print('领域构成同时影响"分组"和"结果"，是典型混杂变量，直接组间比较会得到被污染的结论。')

,数字标题占比,领域平均播放对数,视频数
domain,,,
职场教育,79.4,9.81,591
社科人文,52.8,11.52,1492
科学科普,51.1,12.23,1235
财经商业,39.3,9.70,811


解读：播放量最低的职场教育领域数字标题占比高达 79.4%，而高播放的科学科普仅 51.1%——
领域构成同时影响"分组"和"结果"，是典型混杂变量，直接组间比较会得到被污染的结论。


In [4]:
# 第三步：分领域内对比（控制领域后的条件效应）
rows = []
for dom, sub in df.groupby('domain'):
    g = sub.groupby('title_has_number')['log_view']
    if set(g.groups.keys()) >= {0,1}:
        m0, m1 = g.mean()[0], g.mean()[1]
        rows.append({'知识领域': dom, '不含数字均值': round(m0,3),
                     '含数字均值': round(m1,3), '组内差': round(m1-m0,3)})
display(pd.DataFrame(rows).sort_values('组内差', ascending=False))
print('科学科普领域内数字标题为正（+0.70），其余领域为负——效应存在明显领域异质性，不能一概而论。')

,知识领域,不含数字均值,含数字均值,组内差
1,科学科普,11.870,12.567,0.698
0,社科人文,11.670,11.394,-0.276
3,财经商业,9.829,9.490,-0.339
2,职场教育,10.497,9.625,-0.872


科学科普领域内数字标题为正（+0.70），其余领域为负——效应存在明显领域异质性，不能一概而论。


In [5]:
# 第四步：回归控制混杂（与论文 Model2 同口径，按 uid 聚类稳健标准误）
formula = '''log_view ~ title_has_number + log_fans + log_age + log_duration
+ tag_count + description_len + is_collection + copyright
+ C(pub_month) + C(pub_weekday) + C(domain) + C(level)
+ title_char_len + title_question_type + title_exclamation_type
+ title_sentiment_intensity + curiosity_marker_count + practical_value_word_count
+ image_contrast + image_brightness + image_colorfulness
+ visual_complexity + cover_has_text + cover_text_area_ratio + face_presence'''
m = smf.ols(formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['uid']})
coef, se_c, p_c = m.params['title_has_number'], m.bse['title_has_number'], m.pvalues['title_has_number']
print(f'控制全部混杂后 title_has_number 系数 = {coef:.4f} (SE {se_c:.4f}, p={p_c:.3f})')
print('\n系数由朴素的 -0.109 翻转为 +0.190（p<0.10，边际显著）——这就是辛普森悖论。')
print('结论：在同等账号、领域和内容条件下，数字表达与更好的传播存在正向关联，但观察性数据无法给出严格因果。')

控制全部混杂后 title_has_number 系数 = 0.1904 (SE 0.1098, p=0.083)

系数由朴素的 -0.109 翻转为 +0.190（p<0.10，边际显著）——这就是辛普森悖论。
结论：在同等账号、领域和内容条件下，数字表达与更好的传播存在正向关联，但观察性数据无法给出严格因果。


In [6]:
# 第五步：最小可检测效应（MDE）——当前样本量、80%功效、α=0.05 下能检出多小的效应
z_a, z_b = stats.norm.ppf(0.975), stats.norm.ppf(0.8)
mde = (z_a + z_b) * se
print(f'当前样本可检出的最小均值差（MDE）≈ {mde:.3f}（log_view 单位），约相对变化 {np.expm1(mde):.1%}')
print('控制后的系数 0.190 与 MDE 接近，处于"勉强可检出"区间；要把结论从 p<0.10 提升到 p<0.05，')
print('需要更大样本，或在真实 A/B 实验中通过随机分流消除混杂、提升检验灵敏度。')

当前样本可检出的最小均值差（MDE）≈ 0.220（log_view 单位），约相对变化 24.6%
控制后的系数 0.190 与 MDE 接近，处于"勉强可检出"区间；要把结论从 p<0.10 提升到 p<0.05，
需要更大样本，或在真实 A/B 实验中通过随机分流消除混杂、提升检验灵敏度。


## 实验结论（可直接同步给运营）

1. **朴素对比为负、控制混杂后为正（+0.190，p<0.10）**：这是典型的辛普森悖论，
   说明取数分析必须先厘清混杂结构，不能直接拿分组均值下结论；
2. 在同等账号与领域条件下，数字清单式标题与更好的传播呈正向关联，
   且**科学科普领域**最明显（组内 +0.70），建议优先在该类内容上 A/B 试点；
3. 效应量较小，建议措辞为"**有条件的正向关联，建议小流量实验验证**"，不得宣称已证因果。

## 真实业务 A/B 实验落地检查清单

1. **随机分流**：按用户/曝光请求哈希分组，先做 **AA 测试**验证分流均匀；
2. **样本量前置**：上线前按 MDE 公式反算每组样本量与实验周期，避免"看到差就上线"；
3. **指标体系**：1 个核心指标 + 辅助指标 + **护栏指标**（完播率、取关率等）；
4. **口径冻结**：实验前冻结指标定义与判定标准，杜绝事后挑选指标；
5. **检验方法**：比例指标用两比例 z 检验；连续指标用 Welch t；偏态分布先对数化或用秩和检验；
6. **结果规范**：同时报告效应量、置信区间、显著性与业务意义；
7. **复盘闭环**：全量后跟踪 2–4 周护栏指标，再沉淀为下一轮实验假设。